# Pronóstico de pasajeros con Prophet

## Introducción

El **pronóstico de series temporales** busca estimar valores futuros a partir de observaciones históricas ordenadas por fecha. En este ejercicio estudiaremos el número mensual de pasajeros de una aerolínea.

El problema que queremos resolver es: **¿cuántos pasajeros podemos esperar durante los próximos 12 meses para apoyar decisiones de capacidad, personal y planeación operativa?**

La serie contiene una tendencia de crecimiento y un patrón que se repite cada año. Utilizaremos `Prophet`, una librería diseñada para pronosticar series con tendencia, estacionalidades y cambios graduales en la tendencia. El modelo también proporciona intervalos de incertidumbre, por lo que no solo obtendremos un valor puntual, sino un rango de valores plausibles.

## Objetivos del ejercicio

Al finalizar podremos:

1. Preparar una serie mensual para utilizarla con Prophet.
2. Identificar tendencia y estacionalidad anual.
3. Separar correctamente datos históricos y datos de prueba.
4. Entrenar y evaluar un modelo Prophet.
5. Interpretar la tendencia, la estacionalidad y los intervalos de incertidumbre.
6. Generar un pronóstico para los siguientes 12 meses.

## 1. Instalar y cargar librerías

Google Colab normalmente incluye muchas librerías científicas, pero `prophet` puede requerir instalación explícita. La siguiente celda instala Prophet y las librerías auxiliares. Después importaremos `Prophet`, `pandas`, `seaborn`, `matplotlib` y las métricas de `scikit-learn`.

In [ ]:
%pip install -q prophet seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

## 2. Cargar el dataset de pasajeros

El dataset `flights` de `seaborn` contiene observaciones mensuales de pasajeros entre 1949 y 1960. Es apropiado para este ejercicio porque permite observar crecimiento y estacionalidad anual en una serie corta y fácil de interpretar.

In [ ]:
vuelos = sns.load_dataset('flights')
vuelos.head()

## 3. Preparar los datos para Prophet

Prophet requiere dos columnas con nombres específicos: `ds` para la fecha y `y` para el valor que se desea pronosticar. Convertiremos año y mes en una fecha mensual, ordenaremos los datos y conservaremos únicamente esas dos columnas.

In [ ]:
datos = vuelos.copy()
datos['ds'] = pd.to_datetime(datos['year'].astype(str) + '-' + datos['month'].astype(str) + '-01')
datos = datos[['ds', 'passengers']].rename(columns={'passengers': 'y'}).sort_values('ds')
datos = datos.reset_index(drop=True)

print(f'Periodo: {datos.ds.min():%Y-%m} a {datos.ds.max():%Y-%m}')
print(f'Observaciones: {len(datos)}')
print(f'Valores faltantes: {datos.isna().sum().sum()}')
datos.head()

## 4. Explorar la serie histórica

Antes de modelar conviene observar la forma de la serie. Buscaremos una tendencia ascendente y meses que se comporten de manera parecida año tras año. Estos patrones justifican el uso de una estacionalidad anual en Prophet.

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(datos['ds'], datos['y'], color='#2563eb', linewidth=2)
plt.title('Pasajeros mensuales de la aerolínea')
plt.xlabel('Fecha')
plt.ylabel('Pasajeros')
plt.tight_layout()

## 5. Separar entrenamiento y prueba

Reservaremos los últimos 12 meses como conjunto de prueba. No mezclaremos las observaciones porque en un pronóstico real el modelo solo conoce el pasado al momento de estimar el futuro. Esta separación permite medir el desempeño sobre un periodo que el modelo no utilizó para entrenarse.

In [ ]:
horizonte_prueba = 12
entrenamiento = datos.iloc[:-horizonte_prueba].copy()
prueba = datos.iloc[-horizonte_prueba:].copy()

print(f'Entrenamiento: {entrenamiento.ds.min():%Y-%m} a {entrenamiento.ds.max():%Y-%m}')
print(f'Prueba: {prueba.ds.min():%Y-%m} a {prueba.ds.max():%Y-%m}')

## 6. Entrenar el modelo Prophet

Configuraremos Prophet para datos mensuales: activaremos estacionalidad anual y desactivaremos la semanal y diaria, porque no tienen sentido para observaciones agregadas por mes. `seasonality_mode='multiplicative'` permite que el efecto estacional crezca junto con el nivel de pasajeros.

In [ ]:
modelo = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    interval_width=0.95,
    changepoint_prior_scale=0.05
).fit(entrenamiento)

print('Modelo Prophet entrenado correctamente.')

## 7. Generar el pronóstico del periodo de prueba

Construiremos un dataframe con las fechas de prueba y llamaremos a `predict`. Prophet devolverá `yhat`, que es el pronóstico puntual, además de `yhat_lower` y `yhat_upper`, que forman un intervalo de incertidumbre.

In [ ]:
fechas_prueba = prueba[['ds']].copy()
pronostico_prueba = modelo.predict(fechas_prueba)
resultado_prueba = prueba[['ds', 'y']].merge(
    pronostico_prueba[['ds', 'yhat', 'yhat_lower', 'yhat_upper']],
    on='ds'
)

resultado_prueba.head()

## 8. Evaluar el desempeño

Calcularemos MAE, RMSE y MAPE comparando los valores reales contra `yhat`. Un MAE de 30, por ejemplo, significa que el modelo se alejó en promedio 30 pasajeros. En las tres métricas, valores menores representan mejor desempeño.

In [ ]:
real = resultado_prueba['y']
predicho = resultado_prueba['yhat']
metricas = pd.Series({
    'MAE': mean_absolute_error(real, predicho),
    'RMSE': np.sqrt(mean_squared_error(real, predicho)),
    'MAPE (%)': np.mean(np.abs((real - predicho) / real)) * 100
}, name='Prophet')

metricas.to_frame()

## 9. Interpretar el pronóstico sobre datos no vistos

La gráfica permite revisar si Prophet reproduce el nivel general y los picos estacionales del último año. También muestra el intervalo de incertidumbre: una banda más amplia indica mayor incertidumbre, por lo que la planeación debería considerar escenarios y no únicamente el valor puntual.

In [ ]:
plt.figure(figsize=(13, 6))
plt.plot(entrenamiento['ds'], entrenamiento['y'], label='Entrenamiento', color='#64748b')
plt.plot(prueba['ds'], prueba['y'], label='Real', color='#111827', linewidth=2)
plt.plot(resultado_prueba['ds'], resultado_prueba['yhat'], label='Pronóstico Prophet', color='#16a34a', linewidth=2)
plt.fill_between(resultado_prueba['ds'], resultado_prueba['yhat_lower'], resultado_prueba['yhat_upper'], color='#86efac', alpha=0.35, label='Intervalo 95%')
plt.axvline(prueba['ds'].iloc[0], color='black', linestyle=':', label='Inicio de prueba')
plt.title('Prophet: valores reales y pronóstico sobre el periodo de prueba')
plt.xlabel('Fecha')
plt.ylabel('Pasajeros')
plt.legend()
plt.tight_layout()

## 10. Revisar los componentes del modelo

Prophet permite separar visualmente sus principales componentes. La tendencia muestra el crecimiento estimado; la estacionalidad anual muestra qué meses tienden a estar por encima o por debajo del nivel esperado. Esta descomposición facilita explicar el pronóstico a personas no técnicas.

In [ ]:
fig_componentes = modelo.plot_components(pronostico_prueba)
plt.show()

## Interpretación de los resultados

El modelo debe considerarse útil si mantiene un error razonable frente al volumen de pasajeros y si la gráfica muestra que captura la tendencia y los ciclos anuales. El intervalo de incertidumbre recuerda que una decisión operativa debe contemplar variabilidad.

En este ejercicio no estamos incorporando precios, eventos, capacidad de aeronaves ni cambios externos. Por ello, el resultado representa una extrapolación basada únicamente en el comportamiento histórico de la serie.

## 11. Generar el pronóstico de los próximos 12 meses

Una vez evaluado el modelo, lo ajustaremos nuevamente utilizando toda la historia disponible. Luego generaremos exactamente 12 fechas mensuales futuras. Para datos mensuales utilizamos `freq='MS'`, que representa el inicio de cada mes.

In [ ]:
modelo_final = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    interval_width=0.95,
    changepoint_prior_scale=0.05
).fit(datos)

fechas_futuras = modelo_final.make_future_dataframe(
    periods=12, freq='MS', include_history=False
)
pronostico_futuro = modelo_final.predict(fechas_futuras)

tabla_pronostico = pronostico_futuro[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
tabla_pronostico.columns = ['Mes', 'Pronóstico', 'Límite inferior 95%', 'Límite superior 95%']
tabla_pronostico.round(0)

## 12. Visualizar el pronóstico final

La visualización final muestra la historia completa y los siguientes 12 meses. La banda verde representa el rango de incertidumbre estimado por Prophet y puede utilizarse para construir escenarios de planeación.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(datos['ds'], datos['y'], label='Histórico', color='#2563eb', linewidth=2)
plt.plot(pronostico_futuro['ds'], pronostico_futuro['yhat'], label='Pronóstico próximos 12 meses', color='#16a34a', linewidth=2)
plt.fill_between(pronostico_futuro['ds'], pronostico_futuro['yhat_lower'], pronostico_futuro['yhat_upper'], color='#86efac', alpha=0.35, label='Intervalo 95%')
plt.axvline(datos['ds'].max(), color='black', linestyle=':', label='Último dato disponible')
plt.title('Pronóstico de pasajeros con Prophet')
plt.xlabel('Fecha')
plt.ylabel('Pasajeros')
plt.legend()
plt.tight_layout()

## Conclusiones generales

- Prophet permite construir un pronóstico interpretable a partir de tendencia, estacionalidad e intervalos de incertidumbre.
- La preparación correcta de las columnas `ds` y `y` es indispensable para que el modelo funcione.
- La separación temporal de entrenamiento y prueba permite evaluar el modelo de manera realista.
- En esta serie mensual, la estacionalidad anual es más relevante que la semanal o diaria.
- El valor puntual `yhat` es una referencia central; los límites inferior y superior ayudan a dimensionar escenarios conservadores y optimistas.
- Un pronóstico no sustituye el análisis del negocio: cambios en precios, capacidad, eventos, competencia o restricciones operativas pueden alterar el resultado.
- En un proyecto real convendría comparar Prophet contra modelos base, Holt-Winters y SARIMA, además de realizar validación temporal repetida y monitorear el error después de cada periodo.